# الفصل السادس عشر: الملفات وتنسيقات البيانات

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AhmedAlmaghz/Python-Book/blob/main/dist/notebooks/chapter-16.ipynb)

> **الوقت المتوقع:** ~13 دقيقة قراءة · **11** قسمًا · **38** مثالًا برمجيًا.

**كيف تستخدم هذا الدفتر؟**

1. شغّل الخلايا بالترتيب من الأعلى إلى الأسفل (`Shift` + `Enter`).
2. خلايا الشيفرة قابلة للتعديل: غيّر القيم وأعد التشغيل لتجرّب.
3. خلايا 🧾 معروضة للقراءة فقط (أوامر طرفية، أو إدخال يدوي، أو مقتطف ناقص)، وخلايا ⚠️ تُظهر خطأً مقصودًا يُلتقط تلقائيًا فلا يتوقف «تشغيل الكل».

> **Kaggle:** نزّل الملف من المستودع ثم `File ▸ Import Notebook`، أو استورده مباشرة من GitHub بعد ربط حسابك.

<details>
<summary><b>محتويات الفصل</b></summary>

- **16.1** لماذا يستحق هذا الفصل اهتمامك الكامل؟
- **16.2** الملفات النصية: الأساسيات
- **16.3** `pathlib`: الأسلوب الحديث للتعامل مع المسارات
- **16.4** JSON: لغة تبادل البيانات العالمية
- **16.5** CSV: البيانات الجدولية
- **16.6** أفضل الممارسات
- **16.7** مشروع تطبيقي: مدير جهات اتصال متكامل
- **16.8** تمارين
- **16.9** اختبر فهمك (Quiz)
- **16.10** ملخص الفصل (Cheat Sheet)
- **16.11** ماذا بعد؟

</details>

In [ ]:
#@title ⚙️ تهيئة البيئة  { display-mode: "form" }
import sys, os, random, pathlib

print("Python:", sys.version.split()[0], "|", sys.platform)
assert sys.version_info >= (3, 10), "هذا الكتاب يفترض Python 3.10 أو أحدث"

SEED = 42
random.seed(SEED)

# مجلد عمل معزول حتى لا تتناثر الملفات في جذر الدفتر
WORK = pathlib.Path("book_runtime")
WORK.mkdir(exist_ok=True)
os.chdir(WORK)
print("مجلد العمل:", pathlib.Path.cwd().name)

# ملفات البداية التي يفترضها الفصل (تُكتب دائمًا لضمان نتيجة ثابتة)
pathlib.Path('notes.txt').write_text('ملاحظاتي اليومية\n- تعلّم open() و with\n- تذكّر إغلاق الملف\n', encoding='utf-8')

> ℹ️ يكتب هذا الفصل ملفات على القرص؛ تُنشأ داخل مجلد `book_runtime/`.

## 16.1 لماذا يستحق هذا الفصل اهتمامك الكامل؟

منذ بداية هذا الكتاب، كل برنامج كتبته يعاني من مشكلة واحدة قاتلة: **فقدان الذاكرة التام**. اكتب قائمة مهام، أضف إليها عشرات العناصر، أغلق البرنامج... وكل شيء يختفي. لا أثر، لا سجل، كأن شيئًا لم يكن.

هذا مقبول تمامًا للتمارين التعليمية، لكنه غير مقبول إطلاقًا لأي برنامج حقيقي. تخيّل تطبيق ملاحظات يَنسى كل ملاحظاتك بمجرد إغلاقه، أو برنامج محاسبة يفقد كل الفواتير عند إعادة تشغيل الحاسوب! هذا هو بالضبط سبب وجود **الملفات (Files)**: مساحة تخزين **دائمة** على القرص الصلب، تبقى بياناتك فيها حتى بعد إغلاق البرنامج، بل حتى بعد إطفاء الحاسوب بالكامل.

في هذا الفصل، ستتعلم:
- كيف تقرأ وتكتب ملفات نصية بأمان وبدون تسريب موارد (Resource Leaks).
- كيف تتعامل مع مسارات الملفات والمجلدات بأسلوب حديث وأنيق عبر `pathlib`.
- كيف تتبادل البيانات المُهيكَلة (قوائم، قواميس، كائنات) مع العالم الخارجي عبر **JSON** — التنسيق الأكثر انتشارًا في عالم البرمجة الحديث (تطبيقات الويب، واجهات API، ملفات الإعدادات).
- كيف تتعامل مع البيانات الجدولية (كالتي تراها في Excel) عبر **CSV**.

هذه المهارات هي **حجر الأساس** لكل ما سيأتي لاحقًا في الكتاب: عندما نبني واجهة API في الفصل 19، أو نحلّل بيانات مبيعات في الفصل 20، أو نبني أداة أتمتة في الفصل 21 — كل هذه المسارات تعتمد بشكل أساسي على قراءة وكتابة الملفات بمختلف تنسيقاتها.

> 💡 **نصيحة الخبير:** إتقان هذا الفصل يعني أن برامجك تتحول من «تجارب مؤقتة» إلى «أدوات حقيقية» يمكن الاعتماد عليها. هذا هو الخط الفاصل بين «أتعلم Python» و«أبني برامج فعلية بـ Python».

## 16.2 الملفات النصية: الأساسيات

### فتح ملف وقراءته: الطريقة الآمنة

لفتح ملف في Python، نستخدم الدالة المدمجة `open()`. لكن هناك طريقة **صحيحة** وطريقة **خاطئة وخطيرة** لفعل ذلك.

#### الطريقة الخطيرة (لا تستخدمها أبدًا)

In [ ]:
file = open("notes.txt", "r", encoding="utf-8")
content = file.read()
print(content)
file.close()  # يجب أن نتذكر إغلاق الملف يدويًا!

المشكلة: إن حدث خطأ (استثناء) بين `open()` و `file.close()`، فإن السطر `file.close()` **لن يُنفَّذ أبدًا**، وسيبقى الملف «مفتوحًا» من منظور نظام التشغيل. هذا يُسمى **تسريب موارد (Resource Leak)** — قد يمنع برامج أخرى من الوصول للملف، ويستهلك موارد النظام دون داعٍ، وفي برنامج يفتح آلاف الملفات، قد يتسبب بانهيار كامل للنظام.

#### الطريقة الصحيحة: جملة `with`

In [ ]:
with open("notes.txt", "r", encoding="utf-8") as file:
    content: str = file.read()
    print(content)
# الملف يُغلق تلقائيًا هنا، حتى لو حدث خطأ بالداخل!

تذكر من الفصل الرابع عشر مفهوم **Context Managers** (`__enter__` و `__exit__`)؟ هذا بالضبط ما تستخدمه جملة `with` هنا: `open()` تُرجع كائن ملف يدعم بروتوكول Context Manager، فتضمن جملة `with` استدعاء `file.close()` تلقائيًا بمجرد الخروج من الكتلة، **بغض النظر عمّا حدث بالداخل** (نجاح، أو استثناء، أو `return` مبكر).

> ⚠️ **تحذير قاطع:** استخدم جملة `with` **دائمًا ودون استثناء** عند التعامل مع الملفات. لا يوجد أي سبب مقبول لاستخدام `open()` بدون `with` في كود إنتاجي حديث.

### معاملات `open()` الأساسية

```python
open(file_path, mode, encoding="utf-8")
```

🧾 **للقراءة فقط** — مثال مقتطف يعتمد على `file_path`، `mode` من مثال سابق.

| المعامل | الوصف |
|---|---|
| `file_path` | مسار الملف (نص أو كائن `Path`، سنراه قريبًا) |
| `mode` | وضع الفتح: قراءة، كتابة، إلخ (الجدول التالي) |
| `encoding` | الترميز المستخدم لقراءة/كتابة النص |

### أوضاع الفتح (Modes)

| الوضع | المعنى | إذا لم يوجد الملف | إذا وُجد الملف |
|---|---|---|---|
| `"r"` | قراءة (Read) - الافتراضي | `FileNotFoundError` | يُقرأ محتواه |
| `"w"` | كتابة (Write) | يُنشأ ملف جديد | **يُمحى المحتوى بالكامل!** |
| `"a"` | إضافة (Append) | يُنشأ ملف جديد | يُضاف المحتوى الجديد في النهاية |
| `"x"` | إنشاء حصري (Exclusive) | يُنشأ ملف جديد | `FileExistsError` |
| `"r+"` | قراءة وكتابة | `FileNotFoundError` | قراءة وتعديل بدون محو |
| `"rb"`, `"wb"` | نفس ما سبق، لكن بصيغة بايتات (Binary) | - | - |

> ⚠️ **خطأ شائع وخطير جدًا:** الخلط بين `"w"` و `"a"`. استخدام `"w"` على ملف موجود **يمحو محتواه بالكامل فورًا** بمجرد فتحه، حتى قبل كتابة أي شيء جديد! إن كنت تريد الإضافة إلى محتوى موجود دون فقدانه، استخدم `"a"` حصرًا.

### مثال: الكتابة في ملف

In [ ]:
with open("diary.txt", "w", encoding="utf-8") as file:
    file.write("اليوم الأول: بدأت تعلم Python.\n")
    file.write("كان يومًا ممتعًا جدًا!\n")

بعد تشغيل هذا الكود، سيظهر ملف `diary.txt` في نفس مجلد البرنامج، يحتوي على السطرين بالضبط.

> ⚠️ **تحذير:** `write()` **لا تضيف** سطرًا جديدًا تلقائيًا (بعكس `print()`)! يجب عليك إضافة `\n` يدويًا إن أردت الانتقال لسطر جديد.

### الإضافة إلى ملف موجود

In [ ]:
with open("diary.txt", "a", encoding="utf-8") as file:
    file.write("اليوم الثاني: تعلمت عن الملفات!\n")

الآن يحتوي `diary.txt` على ثلاثة أسطر (السطرين السابقين + السطر الجديد).

### طرق متعددة لقراءة محتوى الملف

In [ ]:
# الطريقة 1: قراءة الملف بالكامل كنص واحد
with open("diary.txt", "r", encoding="utf-8") as file:
    content: str = file.read()
    print(content)

# الطريقة 2: قراءة سطر واحد فقط
with open("diary.txt", "r", encoding="utf-8") as file:
    first_line: str = file.readline()
    print(first_line)  # "اليوم الأول: بدأت تعلم Python.\n"

# الطريقة 3: قراءة كل الأسطر كقائمة
with open("diary.txt", "r", encoding="utf-8") as file:
    lines: list[str] = file.readlines()
    print(lines)
    # ['اليوم الأول: ...\n', 'اليوم الثاني: ...\n', ...]

# الطريقة 4 (الأفضل للملفات الكبيرة): التكرار المباشر سطرًا بسطر
with open("diary.txt", "r", encoding="utf-8") as file:
    for line in file:
        print(line.strip())  # .strip() لإزالة \n الزائدة

> 💡 **نصيحة الخبير - لماذا الطريقة 4 هي الأفضل؟** الطرق 1 و3 تُحمّل **الملف بأكمله** في الذاكرة دفعة واحدة. إن كان الملف بحجم 10 غيغابايت (سجلات خادم مثلاً)، فقد تستهلك كل ذاكرة حاسوبك وينهار البرنامج! التكرار المباشر (`for line in file:`) يقرأ **سطرًا واحدًا في كل مرة** (تمامًا مثل Generators التي تعلمناها في الفصل 12)، فيبقى استهلاك الذاكرة ثابتًا بغض النظر عن حجم الملف.

### الترميز (Encoding): لماذا `encoding="utf-8"` إلزامي دائمًا؟

```python
# ❌ خطير: الاعتماد على الترميز الافتراضي لنظام التشغيل
with open("notes.txt", "r") as file:
    content = file.read()

# ✅ آمن: تحديد الترميز صراحة
with open("notes.txt", "r", encoding="utf-8") as file:
    content = file.read()
```

⚠️ **للقراءة فقط** — خطأ مقصود للتوضيح، لذا لا يُنفَّذ.

> ⚠️ **تحذير حاسم خصوصًا للغة العربية:** إن لم تُحدد `encoding`، فإن Python تستخدم الترميز **الافتراضي لنظام التشغيل**، والذي يختلف بين Windows (غالبًا `cp1252` أو ما شابه) ولينكس وماك (غالبًا `utf-8`). إن كتبت ملفًا يحتوي نصًا عربيًا على نظام ويندوز بدون تحديد `encoding="utf-8"`، فقد يفشل فتحه لاحقًا على نظام آخر، أو تظهر رموز غريبة (مثل `Ø§Ù„Ø¹Ø±Ø¨ÙŠØ©`) بدلًا من النص العربي الصحيح. **القاعدة الذهبية: اكتب `encoding="utf-8"` في كل استدعاء لـ `open()`، بدون استثناء.**

### معالجة أخطاء الملفات

In [ ]:
try:
    with open("missing_file.txt", "r", encoding="utf-8") as file:
        content = file.read()
except FileNotFoundError:
    print("الملف غير موجود!")
except PermissionError:
    print("لا تملك صلاحية الوصول لهذا الملف!")

> 🔄 **مقارنة: EAFP مقابل LBYL (من الفصل 11)**
> ```python
> # أسلوب LBYL (تحقق قبل التنفيذ)
> import os
> if os.path.exists("file.txt"):
>     with open("file.txt", "r", encoding="utf-8") as f:
>         content = f.read()
> else:
>     print(«الملف غير موجود»)
>
> # أسلوب EAFP (الأسلوب الـ Pythonic الموصى به)
> try:
>     with open("file.txt", "r", encoding="utf-8") as f:
>         content = f.read()
> except FileNotFoundError:
>     print(«الملف غير موجود»)
> ```
> الأسلوب الثاني أفضل لأنه **آمن من سباقات التوقيت (Race Conditions)**: قد يُحذف الملف تمامًا في اللحظة الفاصلة بين `os.path.exists()` واستدعاء `open()`!

## 16.3 `pathlib`: الأسلوب الحديث للتعامل مع المسارات

في الإصدارات القديمة من Python، كان التعامل مع مسارات الملفات يتم عبر وحدة `os.path`، باستخدام نصوص عادية ودمجها يدويًا. هذا الأسلوب كان عرضة لمشاكل كثيرة (خاصة اختلاف فاصل المسارات بين Windows `\` ولينكس/ماك `/`).

منذ Python 3.4، أصبح لدينا `pathlib`: وحدة حديثة تُمثّل المسارات ككائنات حقيقية (Objects)، بدلًا من مجرد نصوص. هذا هو **الأسلوب الموصى به حصريًا** في هذا الكتاب.

### إنشاء كائن `Path`

In [ ]:
from pathlib import Path

# مسار نسبي
notes_path = Path("notes.txt")

# مسار بمجلدات متعددة - لاحظ استخدام / كعامل دمج!
project_path = Path("data") / "2024" / "report.txt"
print(project_path)
# على لينكس/ماك: data/2024/report.txt
# على Windows: data\2024\report.txt  (يتكيف تلقائيًا!)

> 🔄 **مقارنة: `os.path` القديمة مقابل `pathlib` الحديثة**
> ```python
> # الأسلوب القديم (os.path) - لا تستخدمه في كود جديد
> import os
> path = os.path.join("data", "2024", "report.txt")
> if os.path.exists(path):
>     print(os.path.getsize(path))
>
> # الأسلوب الحديث (pathlib) - الموصى به
> from pathlib import Path
> path = Path("data") / "2024" / "report.txt"
> if path.exists():
>     print(path.stat().st_size)
> ```
> `pathlib` أكثر قابلية للقراءة، وتعمل بنفس الطريقة على كل أنظمة التشغيل، وتجمع كل الوظائف ذات الصلة بالمسار في كائن واحد متماسك (Object-Oriented)، بدلًا من دوال متفرقة.

### خصائص ودوال مفيدة في `Path`

In [ ]:
from pathlib import Path

p = Path("data/2024/report.txt")

print(p.name)        # "report.txt" - اسم الملف كاملاً
print(p.stem)        # "report" - الاسم بدون الامتداد
print(p.suffix)      # ".txt" - الامتداد فقط
print(p.parent)      # "data/2024" - المجلد الأب
print(p.parts)       # ('data', '2024', 'report.txt') - كل الأجزاء

print(p.exists())    # هل المسار موجود فعليًا؟ True/False
print(p.is_file())   # هل هو ملف؟
print(p.is_dir())    # هل هو مجلد؟
print(p.absolute())  # المسار الكامل المطلق

### إنشاء المجلدات

In [ ]:
from pathlib import Path

data_dir = Path("data/2024")
data_dir.mkdir(parents=True, exist_ok=True)

- `parents=True`: ينشئ كل المجلدات الوسيطة اللازمة (هنا، `data` ثم `2024` بداخله) إن لم تكن موجودة.
- `exist_ok=True`: لا يُصدر خطأ إن كان المجلد موجودًا بالفعل (بدونها، ستحصل على `FileExistsError`).

### قراءة وكتابة سريعة بدون `with` و `open()`

من أجمل ميزات `pathlib`: اختصارات مباشرة لقراءة/كتابة ملفات نصية كاملة، دون الحاجة لـ `open()` يدويًا!

In [ ]:
from pathlib import Path

notes = Path("notes.txt")

# كتابة (تُنشئ الملف تلقائيًا إن لم يكن موجودًا، وتفتح وتغلق تلقائيًا)
notes.write_text("مرحبًا من pathlib!\n", encoding="utf-8")

# قراءة
content: str = notes.read_text(encoding="utf-8")
print(content)

> 💡 **نصيحة الخبير:** استخدم `read_text()`/`write_text()` للملفات البسيطة الصغيرة ذات العملية الواحدة (قراءة كاملة أو كتابة كاملة). استخدم `open()` مع `with` عندما تحتاج تحكمًا أدق (قراءة سطرًا بسطر، الإلحاق، أو عمليات متعددة على نفس الملف المفتوح).

### استكشاف محتويات مجلد

In [ ]:
from pathlib import Path

project_dir = Path(".")  # المجلد الحالي

# كل الملفات والمجلدات مباشرة (دون تعمق)
for item in project_dir.iterdir():
    print(item)

# البحث عن كل ملفات .py في المجلد الحالي فقط
for py_file in project_dir.glob("*.py"):
    print(py_file)

# البحث التكراري في كل المجلدات الفرعية أيضًا
for py_file in project_dir.glob("**/*.py"):
    print(py_file)

# بديل أحدث وأوضح لنفس الغرض (Python 3.12+)
for py_file in project_dir.rglob("*.py"):
    print(py_file)

### مثال عملي: تنظيم ملفات مجلد التنزيلات حسب الامتداد

In [ ]:
from pathlib import Path
import shutil

def organize_downloads(downloads_path: Path) -> None:
    """يُنظّم الملفات في مجلد حسب امتدادها، كل نوع في مجلد فرعي خاص به."""
    extension_folders: dict[str, str] = {
        ".pdf": "مستندات",
        ".jpg": "صور",
        ".png": "صور",
        ".mp4": "فيديوهات",
        ".zip": "أرشيف",
    }

    for file_path in downloads_path.iterdir():
        if file_path.is_file():
            folder_name = extension_folders.get(file_path.suffix.lower())
            if folder_name is not None:
                target_dir = downloads_path / folder_name
                target_dir.mkdir(exist_ok=True)
                shutil.move(str(file_path), str(target_dir / file_path.name))
                print(f"نُقل {file_path.name} إلى {folder_name}/")

organize_downloads(Path.home() / "Downloads")

هذا مثال حقيقي على **أتمتة (Automation)** بسيطة — موضوع سنتعمق فيه في الفصل 21، لكنه يُظهر بوضوح كيف تتحول معرفتك بالملفات إلى أدوات حقيقية موفّرة للوقت.

> 💡 **نصيحة الخبير:** `Path.home()` تُرجع مسار المجلد الرئيسي للمستخدم (`/home/username` على لينكس، أو `C:\Users\username` على ويندوز) بشكل متوافق تلقائيًا مع كل الأنظمة — لا تكتب هذه المسارات يدويًا أبدًا!

## 16.4 JSON: لغة تبادل البيانات العالمية

### ما هو JSON ولماذا هو بهذه الأهمية؟

**JSON** (اختصار JavaScript Object Notation) هو تنسيق نصي بسيط جدًا لتمثيل البيانات المُهيكَلة، أصبح **المعيار العالمي** لتبادل البيانات بين البرامج المختلفة، خاصة في:
- واجهات برمجة التطبيقات (APIs) على الويب (سنستخدمه بكثافة في الفصل 19).
- ملفات الإعدادات (Configuration Files).
- تخزين بيانات بسيطة بدون الحاجة لقاعدة بيانات كاملة.

شكل JSON يُشبه **قواميس وقوائم Python بشكل شبه مطابق**:

```json
{
  "name": "أحمد",
  "age": 30,
  "is_active": true,
  "hobbies": ["قراءة", "برمجة"],
  "address": {
    "city": "القاهرة",
    "zip": "11511"
  },
  "spouse": null
}
```

🧾 **للقراءة فقط** — محتوى ملف `json`.

### جدول المقابلة بين JSON و Python

| JSON | Python |
|---|---|
| `object {}` | `dict` |
| `array []` | `list` |
| `string "..."` | `str` |
| `number` (بدون فاصلة) | `int` |
| `number` (بفاصلة) | `float` |
| `true` / `false` | `True` / `False` |
| `null` | `None` |

### وحدة `json` المدمجة

لا تحتاج تثبيت أي شيء — `json` جزء من المكتبة القياسية (Standard Library) في Python.

In [ ]:
import json

### تحويل Python إلى نص JSON: `dumps()` و `dump()`

In [ ]:
import json

user_data: dict[str, object] = {
    "name": "أحمد",
    "age": 30,
    "is_active": True,
    "hobbies": ["قراءة", "برمجة"],
    "spouse": None,
}

# dumps (مع s) تُرجع نصًا (string) يمكن طباعته أو إرساله
json_string: str = json.dumps(user_data, ensure_ascii=False, indent=2)
print(json_string)

الناتج:

```json
{
  "name": "أحمد",
  "age": 30,
  "is_active": true,
  "hobbies": [
    "قراءة",
    "برمجة"
  ],
  "spouse": null
}
```

🧾 **للقراءة فقط** — محتوى ملف `json`.

- `indent=2`: تنسيق جميل مقروء (بدونها، يُنتج JSON كله في سطر واحد متلاصق — مناسب للتخزين، غير مناسب للقراءة البشرية).
- `ensure_ascii=False`: **مهم جدًا للعربية!** بدونها، ستتحول كل الأحرف العربية إلى رموز Unicode مُرمَّزة (`\u0623\u062d\u0645\u062f`)، غير مقروءة للإنسان إطلاقًا.

> ⚠️ **خطأ شائع جدًا عند التعامل مع العربية في JSON:** نسيان `ensure_ascii=False`. جرّب حذفها وشاهد النتيجة بنفسك — ستتفاجأ بالرموز الغريبة!

### الكتابة المباشرة إلى ملف: `dump()` (بدون s)

In [ ]:
import json
from pathlib import Path

user_data = {"name": "سارة", "age": 25, "city": "دبي"}

with open("user.json", "w", encoding="utf-8") as file:
    json.dump(user_data, file, ensure_ascii=False, indent=2)

> 💡 **نصيحة للتذكر:** `dumps` (بحرف s، اختصار Dump String) تُرجع **نصًا**. `dump` (بدون s) تكتب **مباشرة إلى ملف مفتوح**. نفس المنطق ينطبق على `load`/`loads` في الاتجاه المعاكس.

### القراءة من نص JSON: `loads()`

In [ ]:
import json

json_text = '{"name": "خالد", "age": 28}'
data: dict[str, object] = json.loads(json_text)

print(data["name"])  # "خالد"
print(type(data))    # <class 'dict'>

### القراءة من ملف JSON: `load()` (بدون s)

In [ ]:
import json

with open("user.json", "r", encoding="utf-8") as file:
    data: dict[str, object] = json.load(file)

print(data["name"])  # "سارة"

### مع `pathlib`: اختصار أنيق

In [ ]:
import json
from pathlib import Path

# كتابة
Path("user.json").write_text(
    json.dumps(user_data, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

# قراءة
data = json.loads(Path("user.json").read_text(encoding="utf-8"))

### التعامل مع أخطاء JSON

In [ ]:
import json

corrupted_json = '{"name": "أحمد", "age": }'  # صيغة خاطئة

try:
    data = json.loads(corrupted_json)
except json.JSONDecodeError as e:
    print(f"خطأ في تنسيق JSON: {e}")

### JSON و Dataclasses معًا

تذكر `@dataclass` من الفصل 14. مشكلة: `json.dumps()` **لا تعرف** كيف تحوّل كائن `dataclass` مباشرة إلى JSON:

```python
from dataclasses import dataclass
import json

@dataclass
class Product:
    name: str
    price: float

product = Product("لابتوب", 1200.0)
json.dumps(product)  # ❌ TypeError: Object of type Product is not JSON serializable
```

⚠️ **للقراءة فقط** — خطأ مقصود للتوضيح، لذا لا يُنفَّذ.

#### الحل 1: `dataclasses.asdict()`

In [ ]:
from dataclasses import dataclass, asdict
import json

@dataclass
class Product:
    name: str
    price: float

product = Product("لابتوب", 1200.0)

product_dict: dict[str, object] = asdict(product)
json_string = json.dumps(product_dict, ensure_ascii=False, indent=2)
print(json_string)
# {
#   "name": "لابتوب",
#   "price": 1200.0
# }

#### الحل 2: تحويل عكسي (من JSON إلى Dataclass)

In [ ]:
data = json.loads(json_string)
restored_product = Product(**data)  # فك تغليف القاموس كمعاملات
print(restored_product)  # Product(name='لابتوب', price=1200.0)

> 💡 **نصيحة الخبير:** في مشاريع أكبر (خاصة عند بناء واجهات API في الفصل 19)، ستتعرّف على مكتبة **Pydantic** التي تؤتمت هذا التحويل بالكامل (من وإلى JSON) مع تحقق تلقائي قوي من صحة البيانات أيضًا، باستخدام Type Hints مباشرة. لكن فهم الآلية اليدوية هنا (عبر `asdict` و`**data`) ضروري لفهم ما يحدث «خلف الكواليس».

### مثال متكامل: حفظ واسترجاع قائمة مهام

In [ ]:
from dataclasses import dataclass, asdict
from pathlib import Path
import json

@dataclass
class Task:
    title: str
    done: bool = False

def save_tasks(tasks: list[Task], file_path: Path) -> None:
    data = [asdict(task) for task in tasks]
    file_path.write_text(
        json.dumps(data, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

def load_tasks(file_path: Path) -> list[Task]:
    if not file_path.exists():
        return []
    raw_data = json.loads(file_path.read_text(encoding="utf-8"))
    return [Task(**item) for item in raw_data]

tasks_file = Path("tasks.json")

tasks = [
    Task("تعلم JSON"),
    Task("تعلم CSV", done=True),
]
save_tasks(tasks, tasks_file)

loaded_tasks = load_tasks(tasks_file)
for task in loaded_tasks:
    status = "✅" if task.done else "⬜"
    print(f"{status} {task.title}")

## 16.5 CSV: البيانات الجدولية

### ما هو CSV؟

**CSV** (Comma-Separated Values) هو تنسيق نصي بسيط جدًا لتمثيل بيانات جدولية (صفوف وأعمدة)، تمامًا كجدول Excel. كل سطر يمثل صفًا، والقيم مفصولة بفاصلة:

```csv
name,age,city
أحمد,30,القاهرة
سارة,25,دبي
خالد,28,الرياض
```

🧾 **للقراءة فقط** — محتوى ملف `csv`.

CSV شائع جدًا في: تصدير بيانات من Excel/Google Sheets، تبادل البيانات بين أنظمة محاسبية، وفي تحليل البيانات (سنعود إليه بعمق في الفصل 20 مع pandas).

### وحدة `csv` المدمجة

### الكتابة باستخدام `csv.writer`

In [ ]:
import csv

rows: list[list[str]] = [
    ["name", "age", "city"],
    ["أحمد", "30", "القاهرة"],
    ["سارة", "25", "دبي"],
]

with open("people.csv", "w", encoding="utf-8", newline="") as file:
    writer = csv.writer(file)
    writer.writerows(rows)

> ⚠️ **تحذير مهم:** لاحظ `newline=""` عند فتح الملف للكتابة بصيغة CSV. بدونها، قد تظهر أسطر فارغة إضافية غير مرغوبة على نظام Windows، بسبب اختلاف طريقة تمثيل نهاية السطر بين الأنظمة. هذه قاعدة ثابتة: **دائمًا `newline=""` عند الكتابة بوحدة `csv`**.

### القراءة باستخدام `csv.reader`

In [ ]:
import csv

with open("people.csv", "r", encoding="utf-8") as file:
    reader = csv.reader(file)
    for row in reader:
        print(row)  # كل صف عبارة عن list[str]
# ['name', 'age', 'city']
# ['أحمد', '30', 'القاهرة']
# ['سارة', '25', 'دبي']

> ⚠️ **ملاحظة:** كل القيم المقروءة من CSV هي **نصوص (str) دائمًا**، حتى لو كانت تبدو أرقامًا! يجب تحويلها يدويًا (`int()`, `float()`) إن أردت استخدامها كأرقام فعلية.

### الأسلوب الأفضل: `DictReader` و `DictWriter`

التعامل بالقوائم (`list[str]`) مزعج لأنك تحتاج تذكّر ترتيب الأعمدة بالأرقام (`row[0]`, `row[1]`...). الحل الأفضل: التعامل بالقواميس، حيث يصبح كل صف قاموسًا باسم العمود كمفتاح:

In [ ]:
import csv

# القراءة بـ DictReader
with open("people.csv", "r", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    for row in reader:
        print(row)
        # {'name': 'أحمد', 'age': '30', 'city': 'القاهرة'}
        print(f"{row['name']} يعيش في {row['city']}")

In [ ]:
import csv

# الكتابة بـ DictWriter
people: list[dict[str, str]] = [
    {"name": "منى", "age": "22", "city": "عمّان"},
    {"name": "طارق", "age": "35", "city": "بيروت"},
]

with open("people2.csv", "w", encoding="utf-8", newline="") as file:
    fieldnames = ["name", "age", "city"]
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()   # يكتب صف العناوين تلقائيًا
    writer.writerows(people)

> 💡 **نصيحة الخبير:** استخدم `DictReader`/`DictWriter` دائمًا بدلًا من `reader`/`writer` العاديين، إلا في حالات نادرة جدًا لا تحتاج فيها لأسماء الأعمدة إطلاقًا. الكود أكثر وضوحًا، وأكثر مقاومة للأخطاء عند تغيير ترتيب الأعمدة مستقبلًا.

### مثال متكامل: تحويل بيانات من قائمة Dataclasses إلى CSV

In [ ]:
from dataclasses import dataclass, asdict, fields
import csv
from pathlib import Path

@dataclass
class Employee:
    name: str
    department: str
    salary: float

def export_to_csv(employees: list[Employee], file_path: Path) -> None:
    if not employees:
        return
    fieldnames = [f.name for f in fields(Employee)]
    with open(file_path, "w", encoding="utf-8", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        for emp in employees:
            writer.writerow(asdict(emp))

employees = [
    Employee("ياسمين", "تسويق", 6000.0),
    Employee("عمر", "هندسة", 8500.0),
]

export_to_csv(employees, Path("employees.csv"))

### 🔄 مقارنة شاملة: JSON مقابل CSV

| المعيار | JSON | CSV |
|---|---|---|
| البنية | متداخلة (Nested) - قواميس داخل قوائم داخل قواميس | مسطّحة (Flat) - صفوف وأعمدة فقط |
| الأنواع | يحافظ على الأنواع (رقم، نص، منطقي، null) | كل شيء نص، يحتاج تحويل يدوي |
| القراءة البشرية | جيدة مع `indent` | ممتازة، تفتح مباشرة في Excel |
| الاستخدام الأمثل | إعدادات، APIs، بيانات هرمية معقّدة | جداول بيانات، تصدير/استيراد من Excel |
| الحجم | أكبر نسبيًا (أسماء مفاتيح مكرّرة) | أصغر نسبيًا |

## 16.6 أفضل الممارسات

### استخدم `pathlib` دائمًا بدل نصوص المسارات الخام

```python
# ❌ تجنّب
file_path = "data" + "/" + "2024" + "/" + "report.txt"

# ✅ استخدم
from pathlib import Path
file_path = Path("data") / "2024" / "report.txt"
```

⚠️ **للقراءة فقط** — خطأ مقصود للتوضيح، لذا لا يُنفَّذ.

### تحقق قبل الافتراض

In [ ]:
from pathlib import Path

config_path = Path("config.json")

if config_path.exists():
    config = json.loads(config_path.read_text(encoding="utf-8"))
else:
    config = {}  # قيمة افتراضية آمنة

### استخدم `with` دائمًا، ولا تفتح ملفًا بدونها أبدًا

### حدد `encoding="utf-8"` في كل استدعاء لـ `open()`

### افصل منطق البرنامج عن منطق حفظ/قراءة البيانات

In [ ]:
# ✅ بنية جيدة: دوال واضحة ومسؤولة عن مهمة واحدة
def load_data(path: Path) -> list[dict[str, object]]:
    ...

def save_data(data: list[dict[str, object]], path: Path) -> None:
    ...

def process_data(data: list[dict[str, object]]) -> list[dict[str, object]]:
    ...  # المنطق الفعلي منفصل تمامًا عن I/O

هذا يُسهّل كثيرًا اختبار الكود لاحقًا (الفصل 25)، لأن منطق المعالجة لا يعتمد على وجود ملفات فعلية.

> ⚠️ **خطأ شائع:** محاولة فتح نفس الملف للكتابة (`"w"`) **وهو مفتوح بالفعل للقراءة** في جزء آخر من البرنامج (خاصة على Windows قد يُصدر `PermissionError`). تأكد دائمًا أن جملة `with` السابقة قد انتهت (أي أن الملف أُغلق) قبل فتحه مجددًا بوضع مختلف.

## 16.7 مشروع تطبيقي: مدير جهات اتصال متكامل

لنبني مشروعًا يجمع كل ما تعلمناه: نظام إدارة جهات اتصال يحفظ البيانات بصيغة JSON، ويدعم التصدير إلى CSV، مع Type Hints كاملة ومعالجة أخطاء شاملة.

### المتطلبات

1. كل جهة اتصال: اسم، هاتف، بريد إلكتروني، مجموعة (عائلة/عمل/أصدقاء).
2. حفظ/تحميل تلقائي من ملف JSON.
3. إضافة، بحث، حذف جهات اتصال.
4. تصدير إلى CSV لفتحه في Excel.
5. معالجة الأخطاء بشكل احترافي (ملف غير موجود، JSON تالف، إلخ).

<details>
<summary>✅ الحل النموذجي الكامل</summary>

In [ ]:
"""
مدير جهات الاتصال - مشروع تطبيقي شامل
يغطي: الملفات النصية، pathlib، JSON، CSV، ومعالجة الأخطاء
"""
from __future__ import annotations

from dataclasses import dataclass, asdict, fields
from pathlib import Path
from typing import Literal
import json
import csv

type ContactGroup = Literal["عائلة", "عمل", "أصدقاء", "أخرى"]

# ---------------------------------------------------------
# نموذج البيانات
# ---------------------------------------------------------
@dataclass
class Contact:
    name: str
    phone: str
    email: str = ""
    group: ContactGroup = "أخرى"

    def display(self) -> str:
        return f"📇 {self.name} | {self.phone} | {self.email} | {self.group}"

# ---------------------------------------------------------
# استثناءات مخصصة
# ---------------------------------------------------------
class ContactNotFoundError(Exception):
    """يُرفع عند عدم العثور على جهة اتصال بالاسم المطلوب."""

# ---------------------------------------------------------
# مدير جهات الاتصال
# ---------------------------------------------------------
class ContactManager:
    def __init__(self, storage_path: Path) -> None:
        self.storage_path = storage_path
        self._contacts: list[Contact] = []
        self._load()

    # ----- التحميل والحفظ -----
    def _load(self) -> None:
        """يحمّل جهات الاتصال من ملف JSON إن وُجد، وإلا يبدأ بقائمة فارغة."""
        if not self.storage_path.exists():
            self._contacts = []
            return

        try:
            raw_text = self.storage_path.read_text(encoding="utf-8")
            raw_data = json.loads(raw_text)
            self._contacts = [Contact(**item) for item in raw_data]
        except json.JSONDecodeError:
            print("⚠️ تحذير: ملف البيانات تالف. سيتم البدء بقائمة فارغة.")
            self._contacts = []
        except TypeError:
            print("⚠️ تحذير: بنية البيانات غير متوافقة. سيتم البدء بقائمة فارغة.")
            self._contacts = []

    def save(self) -> None:
        """يحفظ كل جهات الاتصال الحالية إلى ملف JSON."""
        data = [asdict(contact) for contact in self._contacts]
        self.storage_path.write_text(
            json.dumps(data, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )

    # ----- العمليات الأساسية -----
    def add_contact(
        self,
        name: str,
        phone: str,
        email: str = "",
        group: ContactGroup = "أخرى",
    ) -> Contact:
        contact = Contact(name=name, phone=phone, email=email, group=group)
        self._contacts.append(contact)
        self.save()
        return contact

    def find_by_name(self, name: str) -> Contact:
        for contact in self._contacts:
            if contact.name == name:
                return contact
        raise ContactNotFoundError(f"لا توجد جهة اتصال باسم '{name}'")

    def delete_contact(self, name: str) -> None:
        contact = self.find_by_name(name)  # يرفع ContactNotFoundError تلقائيًا إن لم يوجد
        self._contacts.remove(contact)
        self.save()

    def search(self, keyword: str) -> list[Contact]:
        keyword_lower = keyword.lower()
        return [
            c for c in self._contacts
            if keyword_lower in c.name.lower() or keyword_lower in c.phone
        ]

    def get_by_group(self, group: ContactGroup) -> list[Contact]:
        return [c for c in self._contacts if c.group == group]

    def all_contacts(self) -> list[Contact]:
        return self._contacts.copy()

    def count(self) -> int:
        return len(self._contacts)

    # ----- التصدير -----
    def export_to_csv(self, csv_path: Path) -> None:
        if not self._contacts:
            print("لا توجد جهات اتصال للتصدير.")
            return

        fieldnames = [f.name for f in fields(Contact)]
        with open(csv_path, "w", encoding="utf-8", newline="") as file:
            writer = csv.DictWriter(file, fieldnames=fieldnames)
            writer.writeheader()
            for contact in self._contacts:
                writer.writerow(asdict(contact))
        print(f"✅ تم تصدير {len(self._contacts)} جهة اتصال إلى {csv_path}")

    def import_from_csv(self, csv_path: Path) -> int:
        if not csv_path.exists():
            raise FileNotFoundError(f"ملف الاستيراد غير موجود: {csv_path}")

        imported_count = 0
        with open(csv_path, "r", encoding="utf-8") as file:
            reader = csv.DictReader(file)
            for row in reader:
                self._contacts.append(Contact(**row))  # type: ignore[arg-type]
                imported_count += 1
        self.save()
        return imported_count

# ---------------------------------------------------------
# برنامج تجريبي
# ---------------------------------------------------------
def main() -> None:
    manager = ContactManager(Path("contacts.json"))

    manager.add_contact("أحمد علي", "0501234567", "ahmed@example.com", "عمل")
    manager.add_contact("سارة محمد", "0559876543", group="عائلة")
    manager.add_contact("خالد يوسف", "0507654321", "khaled@example.com", "أصدقاء")

    print(f"إجمالي جهات الاتصال: {manager.count()}")
    print()

    print("كل جهات الاتصال:")
    for contact in manager.all_contacts():
        print(contact.display())
    print()

    print("البحث عن 'سارة':")
    for contact in manager.search("سارة"):
        print(contact.display())
    print()

    try:
        manager.find_by_name("غير موجود")
    except ContactNotFoundError as e:
        print(f"خطأ متوقع: {e}")
    print()

    manager.export_to_csv(Path("contacts_export.csv"))

if __name__ == "__main__":
    main()

### فحص المشروع بـ mypy (كما تعلّمنا في الفصل 15)

```bash
uv run mypy contact_manager.py --strict
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

</details>

## 16.8 تمارين

### 🟢 مستوى مبتدئ

**تمرين 1:** اكتب برنامجًا يطلب من المستخدم إدخال 3 أسماء، ثم يحفظها في ملف `names.txt`، كل اسم في سطر منفصل.

**تمرين 2:** اكتب دالة `count_lines(file_path: Path) -> int` تُرجع عدد الأسطر في ملف نصي معيّن.

**تمرين 3:** اكتب برنامجًا يحفظ قاموسًا بسيطًا (اسم، عمر، مدينة) في ملف JSON باسم `person.json`، ثم يقرأه ويطبع محتواه.

<details>
<summary>✅ الحل النموذجي</summary>

```python
# تمرين 1
names: list[str] = [input(f"أدخل الاسم {i + 1}: ") for i in range(3)]

with open("names.txt", "w", encoding="utf-8") as file:
    for name in names:
        file.write(name + "\n")

# تمرين 2
from pathlib import Path

def count_lines(file_path: Path) -> int:
    with open(file_path, "r", encoding="utf-8") as file:
        return sum(1 for _ in file)

print(count_lines(Path("names.txt")))

# تمرين 3
import json
from pathlib import Path

person: dict[str, object] = {"name": "ليلى", "age": 27, "city": "تونس"}

Path("person.json").write_text(
    json.dumps(person, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

loaded = json.loads(Path("person.json").read_text(encoding="utf-8"))
print(loaded)
```

⚠️ **للقراءة فقط** — إدخال تفاعلي داخل حلقة.

</details>


### 🟡 مستوى متوسط

**تمرين 4:** اكتب برنامجًا يقرأ ملف `sales.csv` (أنشئه بنفسك بعمودين: `product`, `amount`)، ويحسب إجمالي المبيعات (مجموع عمود `amount`)، مع معالجة احتمال عدم وجود الملف.

**تمرين 5:** اكتب فئة `Logger` تحتفظ بسجل أحداث، وتوفّر دالة `log(message: str) -> None` تضيف سطرًا جديدًا لملف `app.log` مع الوقت الحالي (استخدم `datetime.now()`)، باستخدام وضع `"a"` (إضافة وليس كتابة).

**تمرين 6:** اكتب دالتين `save_config(config: dict[str, object], path: Path) -> None` و `load_config(path: Path) -> dict[str, object]` للتعامل مع ملف إعدادات JSON، بحيث إن لم يوجد الملف، تُرجع `load_config` قاموسًا افتراضيًا `{"theme": "light", "language": "ar"}`.

<details>
<summary>✅ الحل النموذجي</summary>

In [ ]:
# تمرين 4
import csv
from pathlib import Path

def calculate_total_sales(csv_path: Path) -> float:
    if not csv_path.exists():
        print("ملف المبيعات غير موجود!")
        return 0.0

    total = 0.0
    with open(csv_path, "r", encoding="utf-8") as file:
        reader = csv.DictReader(file)
        for row in reader:
            total += float(row["amount"])
    return total

print(calculate_total_sales(Path("sales.csv")))

# تمرين 5
from pathlib import Path
from datetime import datetime

class Logger:
    def __init__(self, log_path: Path) -> None:
        self.log_path = log_path

    def log(self, message: str) -> None:
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        with open(self.log_path, "a", encoding="utf-8") as file:
            file.write(f"[{timestamp}] {message}\n")

logger = Logger(Path("app.log"))
logger.log("بدأ تشغيل التطبيق")
logger.log("تم تسجيل دخول مستخدم جديد")

# تمرين 6
import json
from pathlib import Path

def save_config(config: dict[str, object], path: Path) -> None:
    path.write_text(
        json.dumps(config, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

def load_config(path: Path) -> dict[str, object]:
    default_config: dict[str, object] = {"theme": "light", "language": "ar"}
    if not path.exists():
        return default_config
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except json.JSONDecodeError:
        return default_config

config = load_config(Path("config.json"))
print(config)
save_config({"theme": "dark", "language": "en"}, Path("config.json"))

</details>


### 🔴 تحدي

**تمرين 7:** ابنِ نظام «نسخ احتياطي» بسيط:
- دالة `backup_file(source: Path, backup_dir: Path) -> Path` تنسخ ملفًا إلى مجلد نسخ احتياطي، مضيفةً الطابع الزمني الحالي إلى اسم النسخة (مثل `notes_20250615_143200.txt`).
- دالة `list_backups(backup_dir: Path, original_name: str) -> list[Path]` تُرجع كل النسخ الاحتياطية الخاصة بملف معيّن، مرتبة من الأحدث إلى الأقدم.
- دالة `restore_latest(backup_dir: Path, original_name: str, destination: Path) -> bool` تستعيد أحدث نسخة احتياطية، وتُرجع `True` إن نجحت، `False` إن لم توجد أي نسخة.
- استخدم `pathlib` حصريًا، و Type Hints كاملة، وتأكد من مرور الكود بـ mypy بوضع `--strict`.

<details>
<summary>✅ الحل النموذجي</summary>

In [ ]:
from pathlib import Path
from datetime import datetime
import shutil

def backup_file(source: Path, backup_dir: Path) -> Path:
    if not source.exists():
        raise FileNotFoundError(f"الملف الأصلي غير موجود: {source}")

    backup_dir.mkdir(parents=True, exist_ok=True)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    backup_name = f"{source.stem}_{timestamp}{source.suffix}"
    backup_path = backup_dir / backup_name

    shutil.copy2(source, backup_path)
    return backup_path

def list_backups(backup_dir: Path, original_name: str) -> list[Path]:
    if not backup_dir.exists():
        return []

    stem = Path(original_name).stem
    suffix = Path(original_name).suffix

    matches = list(backup_dir.glob(f"{stem}_*{suffix}"))
    return sorted(matches, key=lambda p: p.stat().st_mtime, reverse=True)

def restore_latest(backup_dir: Path, original_name: str, destination: Path) -> bool:
    backups = list_backups(backup_dir, original_name)
    if not backups:
        return False

    latest_backup = backups[0]
    shutil.copy2(latest_backup, destination)
    return True

# تجربة
source_file = Path("notes.txt")
source_file.write_text("محتوى تجريبي أول", encoding="utf-8")

backup_dir = Path("backups")

backup_file(source_file, backup_dir)

import time
time.sleep(1)  # للتأكد من اختلاف الطابع الزمني بين النسختين

source_file.write_text("محتوى تجريبي ثانٍ - مُحدّث", encoding="utf-8")
backup_file(source_file, backup_dir)

backups = list_backups(backup_dir, "notes.txt")
print(f"عدد النسخ الاحتياطية: {len(backups)}")
for b in backups:
    print(f"  {b.name}")

success = restore_latest(backup_dir, "notes.txt", Path("restored_notes.txt"))
print(f"تمت الاستعادة بنجاح: {success}")
print(Path("restored_notes.txt").read_text(encoding="utf-8"))

تحقق:

```bash
uv run mypy backup_system.py --strict
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

</details>

## 16.9 اختبر فهمك (Quiz)

<details>
<summary>❓ السؤال 1: لماذا نستخدم جملة `with` عند فتح الملفات بدلًا من `open()` و `close()` يدويًا؟</summary>

لأن `with` تضمن إغلاق الملف تلقائيًا بمجرد الخروج من الكتلة، **حتى لو حدث استثناء بالداخل**. استخدام `close()` يدويًا يعرّض البرنامج لتسريب الموارد (Resource Leak) إن حدث خطأ قبل الوصول لسطر `close()`.
</details>

<details>
<summary>❓ السؤال 2: ما الفرق بين وضعي الفتح `"w"` و `"a"`؟</summary>

`"w"` (Write) يمحو محتوى الملف بالكامل فور فتحه إن كان موجودًا، ثم يكتب من جديد. `"a"` (Append) يضيف المحتوى الجديد في نهاية الملف دون حذف أي شيء موجود مسبقًا.
</details>

<details>
<summary>❓ السؤال 3: لماذا يجب تحديد `encoding="utf-8"` دائمًا عند فتح الملفات؟</summary>

لأن الترميز الافتراضي يختلف حسب نظام التشغيل، مما قد يتسبب في ظهور رموز غريبة أو أخطاء عند التعامل مع نصوص غير إنجليزية (كالعربية) أو عند نقل الملف بين أنظمة تشغيل مختلفة. تحديد `utf-8` صراحة يضمن سلوكًا متسقًا على كل الأنظمة.
</details>

<details>
<summary>❓ السؤال 4: ما الفرق بين `json.dump()` و `json.dumps()`؟</summary>

`dump()` (بدون s) تكتب البيانات **مباشرة إلى ملف مفتوح**. `dumps()` (بحرف s، اختصار "dump string") تُرجع **نصًا (string)** يمكن طباعته أو التعامل معه يدويًا. نفس المنطق ينطبق على `load()`/`loads()` في الاتجاه المعاكس.
</details>

<details>
<summary>❓ السؤال 5: لماذا نحتاج معامل `newline=""` عند فتح ملف CSV للكتابة؟</summary>

لتجنّب ظهور أسطر فارغة إضافية غير مرغوبة، خاصة على نظام Windows، بسبب اختلاف طريقة تمثيل «نهاية السطر» بين أنظمة التشغيل. وحدة `csv` تتولى التعامل مع نهايات الأسطر بنفسها بشكل صحيح فقط عند تحديد `newline=""`.
</details>

<details>
<summary>❓ السؤال 6: لماذا تُعتبر `pathlib` أفضل من `os.path` في الكود الحديث؟</summary>

لأنها تمثّل المسارات ككائنات حقيقية (Objects) بدلًا من نصوص خام، توفّر عامل `/` أنيقًا لدمج المسارات، تتكيّف تلقائيًا مع اختلاف فاصل المسارات بين أنظمة التشغيل، وتجمع كل الوظائف ذات الصلة (فحص الوجود، القراءة، الكتابة، استخراج الاسم/الامتداد) في واجهة واحدة متماسكة وأكثر قابلية للقراءة.
</details>

## 16.10 ملخص الفصل (Cheat Sheet)

```python
# ===== فتح الملفات النصية =====
with open("file.txt", "r", encoding="utf-8") as f:
    content = f.read()           # الملف كاملاً كنص
    # أو
    lines = f.readlines()        # قائمة بكل الأسطر
    # أو (الأفضل للملفات الكبيرة)
    for line in f:
        print(line.strip())

with open("file.txt", "w", encoding="utf-8") as f:
    f.write("نص جديد\n")         # يمحو المحتوى القديم!

with open("file.txt", "a", encoding="utf-8") as f:
    f.write("إضافة\n")           # يُضيف دون حذف

# ===== pathlib =====
from pathlib import Path

p = Path("data") / "file.txt"
p.exists()          # هل موجود؟
p.is_file()         # هل ملف؟
p.mkdir(parents=True, exist_ok=True)
p.write_text("محتوى", encoding="utf-8")
content = p.read_text(encoding="utf-8")
p.name, p.stem, p.suffix, p.parent

for f in Path(".").glob("*.py"):
    print(f)

# ===== JSON =====
import json

json.dumps(data, ensure_ascii=False, indent=2)   # Python → نص JSON
json.loads(json_text)                             # نص JSON → Python
json.dump(data, file, ensure_ascii=False, indent=2)  # Python → ملف
json.load(file)                                   # ملف → Python

# مع Dataclasses
from dataclasses import asdict
json.dumps(asdict(my_object), ensure_ascii=False)

# ===== CSV =====
import csv

with open("f.csv", "w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=["a", "b"])
    writer.writeheader()
    writer.writerows(data)

with open("f.csv", "r", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    for row in reader:
        print(row)  # dict[str, str]
```

🧾 **للقراءة فقط** — مثال مقتطف يعتمد على `my_object` من مثال سابق.

| المهمة | الأداة الموصى بها |
|---|---|
| مسارات الملفات | `pathlib.Path` |
| نص بسيط كامل | `Path.read_text()` / `write_text()` |
| قراءة تدريجية (ملفات كبيرة) | `with open(...) as f: for line in f:` |
| بيانات هرمية (إعدادات، APIs) | `json` |
| بيانات جدولية (Excel) | `csv.DictReader` / `DictWriter` |
| معالجة أخطاء الملفات | `try/except FileNotFoundError` |

## 16.11 ماذا بعد؟

الآن برامجك لم تعد «تُنسى» بمجرد إغلاقها — أصبحت قادرة على حفظ بياناتها، قراءتها، وتبادلها مع العالم الخارجي بأكثر التنسيقات انتشارًا. هذه المهارة ستكون حاضرة في كل مشروع تقريبًا من الآن فصاعدًا، في كل مسارات التخصص القادمة.

لكن هناك تحدٍ آخر ينتظرنا: ماذا لو احتاج برنامجك لتحميل 100 ملف من الإنترنت، أو استدعاء واجهة API عدة مرات، أو قراءة عدة ملفات ضخمة في آن واحد؟ لو نفّذنا هذه العمليات بالطريقة التقليدية (Synchronous) التي تعلمناها حتى الآن، سينتظر برنامجك كل عملية حتى تنتهي تمامًا قبل الانتقال للتالية — مما يُهدر وقتًا ثمينًا بلا داعٍ.

في الفصل السابع عشر، سندخل عالم **البرمجة غير المتزامنة (Asynchronous Programming)**: مفهوم `async`/`await`، لماذا هو ضروري خصوصًا في برمجة الويب والشبكات، وكيف يسمح لبرنامجك بتنفيذ مهام متعددة «في نفس الوقت تقريبًا» دون الحاجة لتعقيدات الـ Threading التقليدية.

---

### الخطوة التالية

- [الفصل السابع عشر: البرمجة غير المتزامنة - مقدمة عملية (async/await)](https://colab.research.google.com/github/AhmedAlmaghz/Python-Book/blob/main/dist/notebooks/chapter-17.ipynb)
- [فهرس كل الدفاتر](https://colab.research.google.com/github/AhmedAlmaghz/Python-Book/blob/main/dist/notebooks/00-index.ipynb)

*الفصل السادس عشر — من كتاب «بايثون 3.14: من الصفر إلى الاحتراف».*